In [5]:
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt 

In [6]:
def calculate_average_mean_TT(summary_file, latest_step):
    tree = ET.parse(summary_file)
    root = tree.getroot()

    mtt_values = [
        float(step.attrib.get("meanTravelTime", "-1"))
        for step in root.findall("step")
    ]
    mtt_array = np.array(mtt_values)

    # filter until lastest step where our controlled vehicle reach dest
    mtt_array = mtt_array[0:latest_step]
    # Filter out -1
    valid_mtt = mtt_array[mtt_array > 0]

    if valid_mtt.size > 0:
        return np.mean(valid_mtt)
    else:
        return None

# Manhattan duration


In [7]:
def get_duration_trips(tripinfo_file):

    # Parse tripinfo.xml and extract arrival times for trips starting with "trip_"
    tripinfo_tree = ET.parse(tripinfo_file)
    tripinfo_root = tripinfo_tree.getroot()

    durations_time = []
    arrival_times = []
    for t in tripinfo_root.findall('tripinfo'):
        trip_id = t.get('id')
        if trip_id.startswith("trip_"):
            duration = t.get('duration')
            arrival = t.get('arrival')
            if duration is not None:
                durations_time.append(float(duration))
                arrival_times.append(float(arrival))

    print("duration times:\n", sorted(durations_time))
    print()
    print("MTT:", np.average(durations_time))
    print("network clearance time", list(sorted(arrival_times))[-1])
    print("max travel time", max(durations_time))
 

In [4]:
## trip info for R1 
# tripinfo_file = r"d:\PhD_Life\Work\Code\routing-MARL\network\toynet2\tripinfo.xml"
tripinfo_file = r"d:\PhD_Life\Work\Code\routing-MARL\network\manhattan\tripinfo.xml"
get_duration_trips(tripinfo_file)

duration times:
 []

MTT: nan


d:\Install\Anaconda\space\envs\disa-simulation\lib\site-packages\numpy\lib\function_base.py:518: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
d:\Install\Anaconda\space\envs\disa-simulation\lib\site-packages\numpy\core\_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


IndexError: list index out of range

In [19]:
## trip info for A*
tripinfo_file = r"d:\PhD_Life\Work\Code\routing-MARL\network\manhattan\tripinfo.xml"
get_duration_trips(tripinfo_file)

duration times:
 [143.0, 153.0, 153.0, 158.0, 180.0, 296.0, 305.0, 386.0, 406.0, 431.0]

MTT: 261.1
network clearance time 436.0
max travel time 431.0


In [9]:
import subprocess
import sys
import os

if 'SUMO_HOME' not in os.environ:
    sys.exit("Please set the SUMO_HOME environment variable")

# Get the path to the directory containing duaIterate.py
# dua_iterate_dir = os.path.join(os.environ['SUMO_HOME'], 'tools', 'assign')
dua_iterate_path = "SUMO_DUE/duaIterate.py"

tools_path = os.path.join(os.environ['SUMO_HOME'], 'tools')
sys.path.append(tools_path)

def run_dua_iterate(sumo_net_path, trips_path):
    cmd = [
        "python",
        dua_iterate_path,
        "-n", sumo_net_path,
        "-t", trips_path,
        "-l", "1000" # 1000 iterations for big network
        # "--convergence-steps", "10"
    ]

    # "--time-to-teleport", "1800",
    # "--tls.all-off", "true",
    # "--no-internal-links", "true"

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print("Command output:\n", result.stdout)
        print("Command errors (if any):\n", result.stderr)
    except subprocess.CalledProcessError as e:
        print(f"Error running command: {e}")
        print("Output:\n", e.output)
        print("Error output:\n", e.stderr)

# Run Dynamic User Assignment (assume follow Dynamic User Equilirbirum)
# sumo_net_path = "../network/toynet2/net2.net.xml"
# trips_path = "../network/toynet2/net2.rou.xml"

sumo_net_path = "../network/manhattan/manhattan.net.xml"
trips_path = "../network/manhattan/manhattan.rou.xml"

run_dua_iterate(sumo_net_path, trips_path)

Command output:
 > Executing step 0
>> Running router on ../network/manhattan/manhattan.rou.xml
>>> Begin time: 2025-06-28 23:35:24.683470
>>> End time: 2025-06-28 23:35:24.958453
>>> Duration: 0:00:00.274983
<<
>> Running simulation
>>> Begin time: 2025-06-28 23:35:24.958453
>>> End time: 2025-06-28 23:35:25.490051
>>> Duration: 0:00:00.531598
<<
< Step 0 ended (duration: 0:00:00.806581)
------------------

> Executing step 1
>> Running router on 000/manhattan_000.rou.alt.xml
>>> Begin time: 2025-06-28 23:35:25.494747
>>> End time: 2025-06-28 23:35:25.776619
>>> Duration: 0:00:00.281872
<<
>> Running simulation
>>> Begin time: 2025-06-28 23:35:25.776619
>>> End time: 2025-06-28 23:35:26.179450
>>> Duration: 0:00:00.402831
<<
< Step 1 ended (duration: 0:00:00.685658)
------------------

> Executing step 2
>> Running router on 001/manhattan_001.rou.alt.xml
>>> Begin time: 2025-06-28 23:35:26.182400
>>> End time: 2025-06-28 23:35:26.582708
>>> Duration: 0:00:00.400308
<<
>> Running simul

In [6]:
## trip info for SUMO
print("SUMO TRIP INFO")
# tripinfo_file = r"D:\PhD_Life\Work\Code\routing-MARL\analyse\KPI\manhattan_notraffic\33.xml"
tripinfo_file = "999/tripinfo_999.xml"
get_duration_trips(tripinfo_file)

SUMO TRIP INFO
duration times:
 [35.0, 37.0, 39.0, 39.0]

MTT: 37.5
network clearance time 48.0
max travel time 39.0


# Summaryze KPI


## No Traffic

In [36]:
kpi_path_no = 'KPI/manhattan_notraffic/'

In [37]:
trip_info_1 = kpi_path_no + 'trip_r1.xml'
latest_trip_id1, latest_arrival1 = get_latest_trip_from_file(trip_info_1)
print(latest_trip_id1, latest_arrival1)


latest_step = int(latest_arrival1) +1

# # travel time for R1 only; no congestion; no disaster
summary_file_1= kpi_path_no + 'r1.xml'
MTT_1= calculate_average_mean_TT(summary_file_1, latest_step)
print("MTT = ", MTT_1)


 trip trip_4 arrvive 98.0
 trip trip_6 arrvive 103.0
 trip trip_0 arrvive 145.0
 trip trip_2 arrvive 149.0
 trip trip_1 arrvive 157.0
 trip trip_8 arrvive 170.0
 trip trip_7 arrvive 181.0
 trip trip_5 arrvive 183.0
 trip trip_9 arrvive 189.0
trip_9 189.0
MTT =  105.35434782608692


In [41]:
# file 2: travel time for A* offline planning

trip_info_2 = kpi_path_no + 'trip_astar.xml'
latest_trip_id2, latest_arrival2 = get_latest_trip_from_file(trip_info_2)
print(latest_trip_id2, latest_arrival2)

latest_step_2 = int(latest_arrival2) + 1

summary_file_2 = kpi_path_no + 'astar.xml'
MTT_2 = calculate_average_mean_TT(summary_file_2, latest_step_2)
print("MTT = ", MTT_2)

 trip trip_0 arrvive 141.0
 trip trip_3 arrvive 147.0
 trip trip_4 arrvive 157.0
 trip trip_6 arrvive 162.0
 trip trip_7 arrvive 182.0
 trip trip_2 arrvive 315.0
 trip trip_1 arrvive 336.0
 trip trip_8 arrvive 351.0
 trip trip_9 arrvive 386.0
 trip trip_5 arrvive 388.0
trip_5 388.0
MTT =  160.48129032258063


In [24]:
# travel time for Dynamic User Assignment (assume follow Dynamic User Equilirbirum)
trip_info_3 = kpi_path_no + 'trip_sumo.xml'
latest_trip_id3, latest_arrival3 = get_latest_trip_from_file(trip_info_3)
print(latest_trip_id3, latest_arrival3)

latest_step_3 = int(latest_arrival3) + 1

summary_file_3 = kpi_path_no + 'sumo.xml'
MTT_3 = calculate_average_mean_TT(summary_file_3, latest_step_3)
print("MTT = ", MTT_3)

 trip trip_0 arrvive 143.0
 trip trip_3 arrvive 145.0
 trip trip_4 arrvive 157.0
 trip trip_6 arrvive 162.0
 trip trip_7 arrvive 180.0
 trip trip_2 arrvive 315.0
 trip trip_1 arrvive 339.0
 trip trip_8 arrvive 350.0
 trip trip_5 arrvive 387.0
trip_5 387.0
MTT =  159.9224897959184


## Traffic

In [26]:
kpi_path = 'KPI/manhattan_traffic/'

In [27]:
trip_info_1 = kpi_path + 'trip_r1.xml'
latest_trip_id1, latest_arrival1 = get_latest_trip_from_file(trip_info_1)
print(latest_trip_id1, latest_arrival1)


latest_step = int(latest_arrival1) +1

# # travel time for R1 only; no congestion; no disaster
summary_file_1= kpi_path + 'r1.xml'
MTT_1= calculate_average_mean_TT(summary_file_1, latest_step)
print("MTT = ", MTT_1)


 trip trip_4 arrvive 105.0
 trip trip_6 arrvive 107.0
 trip trip_2 arrvive 148.0
 trip trip_9 arrvive 151.0
 trip trip_8 arrvive 156.0
 trip trip_0 arrvive 166.0
 trip trip_7 arrvive 177.0
 trip trip_1 arrvive 184.0
 trip trip_3 arrvive 186.0
 trip trip_5 arrvive 190.0
trip_5 190.0
MTT =  113.05872093023255


In [28]:
# file 2: travel time for A* offline planning

trip_info_2 = kpi_path + 'trip_astar.xml'
latest_trip_id2, latest_arrival2 = get_latest_trip_from_file(trip_info_2)
print(latest_trip_id2, latest_arrival2)

latest_step_2 = int(latest_arrival2) + 1

summary_file_2 = kpi_path + 'astar.xml'
MTT_2 = calculate_average_mean_TT(summary_file_2, latest_step_2)
print("MTT = ", MTT_2)

 trip trip_0 arrvive 141.0
 trip trip_7 arrvive 159.0
 trip trip_4 arrvive 170.0
 trip trip_6 arrvive 172.0
 trip trip_3 arrvive 174.0
trip_3 174.0
MTT =  141.8323529411765


In [29]:
# travel time for Dynamic User Assignment (assume follow Dynamic User Equilirbirum)
trip_info_3 = kpi_path + 'trip_sumo.xml'
latest_trip_id3, latest_arrival3 = get_latest_trip_from_file(trip_info_3)
print(latest_trip_id3, latest_arrival3)

latest_step_3 = int(latest_arrival3) + 1

summary_file_3 = kpi_path + 'sumo.xml'
MTT_3 = calculate_average_mean_TT(summary_file_3, latest_step_3)
print("MTT = ", MTT_3)

 trip trip_0 arrvive 145.0
 trip trip_3 arrvive 147.0
 trip trip_4 arrvive 173.0
 trip trip_7 arrvive 182.0
 trip trip_6 arrvive 186.0
 trip trip_1 arrvive 302.0
 trip trip_8 arrvive 349.0
 trip trip_5 arrvive 355.0
 trip trip_9 arrvive 359.0
 trip trip_2 arrvive 375.0
trip_2 375.0
MTT =  149.7378354978355
